# SemanticPromptTransfer v0.26.5 — Gemma 4 native vLLM 심사지원 에이전트 운영

위에서부터 모든 셀을 실행합니다. 로그인과 ngrok 공통 비밀번호 없이 HTML이 바로 열립니다. 브라우저별 난수 ID로 파일·벡터·생성 작업을 분리합니다.

첨부된 우수 심사역 FEW SHOT 1~3이 기본값으로 입력되어 있으며 수정할 수 있습니다. 신용조사서는 선택사항이고, 미첨부 시 사업보고서 등 첨부자료만으로 RAG를 구성합니다.

Gemma 4 26B-A4B MoE는 native vLLM 구현으로 고정하고, vLLM과 Ninja를 같은 격리환경에 설치합니다. A100 80GB BF16 설정에서 최대 4개 요청을 연속 배칭합니다. Colab Secrets 필수값은 `NGROK_AUTHTOKEN`, `HF_TOKEN` 두 개뿐입니다.


v0.26.5는 검증 실패를 자동보정·근거기반 대체생성으로 수렴시키며, 신용조사서와 기타 첨부자료는 상시 우선순위 없이 통합 검토합니다. 동일 사실이 직접 충돌할 때만 신용조사서를 채택합니다.

v0.26.5는 각 익명 테스트 세션에 ABC기업 신용조사서와 사업보고서를 UPLOADED 상태로 최초 1회 복제합니다. 파일명 클릭 다운로드와 × 삭제가 가능하며, 심사의견 생성 전에는 파싱·벡터 임베딩을 수행하지 않습니다.


In [ ]:
"""Single-cell Google Colab launcher for SemanticPromptTransfer v0.26.5.

The notebook generated from this source mounts the owner's Google Drive,
verifies and stages the approved package wheel under /content, starts the
FastAPI application and packaged HTML on one port, and exposes one ngrok URL
without application or tunnel login. User uploads and vectors never write back to
Google Drive.
"""

import atexit
import gzip
import hashlib
import json
import shutil
import subprocess
import sys
import threading
import time
from getpass import getpass
from pathlib import Path
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen


RELEASE = "v0.26.5"
PACKAGE_VERSION = "0.26.5"
DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "SemanticPromptTransfer"
ASSET_MANIFEST_CANDIDATES = (
    DRIVE_ROOT / "runtime-assets" / RELEASE / f"SemanticPromptTransfer_{RELEASE}_COLAB_ASSETS.json",
    DRIVE_ROOT / "versions" / RELEASE / f"SemanticPromptTransfer_{RELEASE}_COLAB_ASSETS.json",
)
ASSET_MANIFEST = ASSET_MANIFEST_CANDIDATES[0]
PORT = 8000
DEMO_ROOT = DRIVE_ROOT / "demo-assets"
DEMO_CREDIT_REPORT = DEMO_ROOT / "신용조사서_ABC기업_v1.0.xlsx"
DEMO_ATTACHMENTS = (DEMO_ROOT / "[ABC기업]사업보고서(2026.03.23).pdf",)


def log(message: str) -> None:
    print(f"[SemanticPromptTransfer] {message}", flush=True)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_reset(path: Path, expected: str) -> None:
    resolved = path.resolve()
    if resolved.parent != Path("/content") or resolved.name != expected:
        raise RuntimeError(f"unsafe Colab cleanup target: {resolved}")
    if resolved.exists():
        shutil.rmtree(resolved)
    resolved.mkdir(parents=True, exist_ok=False)


def secret(name: str, *, prompt: str | None = None, default: str = "") -> str:
    value = ""
    try:
        from google.colab import userdata

        value = str(userdata.get(name) or "").strip()
    except Exception:
        value = ""
    if not value and prompt:
        value = getpass(prompt).strip()
    return value or default


def stage_assets(manifest: dict, stage_root: Path) -> dict[str, Path]:
    staged: dict[str, Path] = {}
    for item in manifest["assets"]:
        source = (DRIVE_ROOT / item["source"]).resolve()
        target = (stage_root / item["target"]).resolve()
        if not source.is_relative_to(DRIVE_ROOT.resolve()):
            raise RuntimeError(f"asset escaped Drive root: {source}")
        if not target.is_relative_to(stage_root.resolve()):
            raise RuntimeError(f"asset escaped stage root: {target}")
        if not source.is_file():
            raise FileNotFoundError(f"Drive asset is missing: {source}")
        if source.stat().st_size != int(item["size"]):
            raise RuntimeError(f"Drive asset size mismatch: {source.name}")
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        if target.stat().st_size != int(item["size"]):
            raise RuntimeError(f"staged asset size mismatch: {target.name}")
        if sha256(target) != item["sha256"]:
            raise RuntimeError(f"staged asset SHA-256 mismatch: {target.name}")
        staged[item["role"]] = target
        log(f"verified: {item['role']} ({target.stat().st_size:,} bytes)")
    return staged


def materialize_model(
    manifest: dict, staged: dict[str, Path], stage_root: Path
) -> Path:
    archive = staged["model_gzip"]
    specification = manifest["model_output"]
    target = (stage_root / specification["target"]).resolve()
    if not target.is_relative_to(stage_root.resolve()):
        raise RuntimeError(f"model output escaped stage root: {target}")
    target.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(archive, "rb") as source, target.open("wb") as destination:
        shutil.copyfileobj(source, destination, length=1024 * 1024)
    if target.stat().st_size != int(specification["size"]):
        raise RuntimeError("decompressed model size mismatch")
    if sha256(target) != specification["sha256"]:
        raise RuntimeError("decompressed model SHA-256 mismatch")
    staged["model"] = target
    log(f"verified: model output ({target.stat().st_size:,} bytes)")
    return target


def install_runtime(wheel: Path) -> None:
    requirement = f"semantic-prompt-transfer[poc] @ {wheel.resolve().as_uri()}"
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        requirement,
        "pyngrok==8.1.2",
    ]
    log("installing package and Colab web dependencies")
    subprocess.run(command, check=True)


previous_cleanup = globals().get("_SPT_COLAB_CLEANUP")
if callable(previous_cleanup):
    log("closing the previous POC process before restart")
    previous_cleanup()

from google.colab import drive

log("mounting Google Drive")
drive.mount(str(DRIVE_MOUNT), force_remount=False)

ASSET_MANIFEST = next(
    (path for path in ASSET_MANIFEST_CANDIDATES if path.is_file()),
    ASSET_MANIFEST_CANDIDATES[0],
)
if not ASSET_MANIFEST.is_file():
    checked = "\n - ".join(str(path) for path in ASSET_MANIFEST_CANDIDATES)
    raise FileNotFoundError(
        "Colab asset manifest is missing. Checked:\n - " + checked
    )
log(f"using asset manifest: {ASSET_MANIFEST}")
manifest = json.loads(ASSET_MANIFEST.read_text(encoding="utf-8"))
if manifest.get("package_version") != PACKAGE_VERSION:
    raise RuntimeError("asset manifest package version mismatch")

for demo_path in (DEMO_CREDIT_REPORT, *DEMO_ATTACHMENTS):
    if not demo_path.is_file():
        raise FileNotFoundError(f"demo asset is missing: {demo_path}")
log("ABC기업 demo assets ready · deferred processing")

stage_root = Path(manifest["stage_root"])
runtime_root = Path(manifest["runtime_root"])
safe_reset(stage_root, "spt_bootstrap_v0265")
staged = stage_assets(manifest, stage_root)
install_runtime(staged["wheel"])

log("installing Korean document font")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"], check=True)



In [ ]:
# FEW SHOT 1 — 실제 우수 심사역 답안 기본값
# 모든 여신유형·업종에 일괄 적용하며, 사실이 아닌 문체·분석 구조로만 사용합니다.
FEW_SHOT_1 = {'case_summary': '[FEW SHOT 사례 / 현재 심사건의 사실 근거가 아님]\n'
                 '\n'
                 '금액 단위는 별도 표시가 없는 한 백만원임.\n'
                 '\n'
                 '■ A. 재무제표 주요계정 관련 기초자료\n'
                 '\n'
                 '[주요계정]\n'
                 '                          2023-12    2024-12    2025-12    추정1기\n'
                 '매출채권                   136,281      85,266     131,313     152,688\n'
                 '재고자산                   195,192     265,187     325,059     280,946\n'
                 '매입채무                    68,606      57,629      91,594      75,489\n'
                 '유형자산                   275,636     274,641     276,453     327,172\n'
                 '총차입금                   약260,352    313,642     314,398     313,856\n'
                 '자본총계                   247,040     257,523     318,022     398,251\n'
                 '매출액                     596,940     687,959     744,818     842,837\n'
                 '영업이익                    16,562      37,373      64,753      76,089\n'
                 '금융비용                    13,672      17,059      12,462      12,471\n'
                 '감가상각비                   8,099       8,664       9,375       1,426\n'
                 '\n'
                 '[특이사항]\n'
                 '- 매출액 대비 매출채권 및 재고자산 과다\n'
                 '- 차입금 중 일반대 증가\n'
                 '- 자본총계 항목 변동\n'
                 '\n'
                 '[재고 관련 추가자료]\n'
                 '- 2024년말 재공품: 121,995\n'
                 '- 2024년말 미착원재료: 58,030\n'
                 '- 제작기간이 장기간인 사업 특성이 존재\n'
                 '- 당기 중 감모손실 반영 금액 없음\n'
                 '\n'
                 '[차입금 계정별 잔액]\n'
                 '                          2023년말    2024년말    2025년말\n'
                 '내국수입유산스               102,404     158,116      74,263\n'
                 '운전대                       156,900     154,200     239,350\n'
                 '기타 금융부채                  1,048       1,326         785\n'
                 '합계                         260,352     313,642     314,398\n'
                 '\n'
                 '- 2024년말 금융차입금은 2023년말 대비 증가\n'
                 '- 2025년말 금융차입금도 2024년말 대비 증가\n'
                 '- 유산스 잔액에 따라 차입금 변동성이 존재\n'
                 '- 2025년에는 수주 증가에 따른 운전자금 충당 목적 등의 일반대 증가가 나타남\n'
                 '\n'
                 '[자본 변동 관련자료]\n'
                 '- 2025.03월 최대주주의 영구전환사채 보통주 전환\n'
                 '- 전환에 따라 신종자본증권 94,351 감소\n'
                 '- 자본금 31,230 증가\n'
                 '- 주식발행초과금 62,966 증가\n'
                 '- 2025.03월 자본잉여금으로 이익잉여금(결손금) 405,326 보전\n'
                 '- 순이익 64,566의 자본 반영 등으로 자본총계 증가\n'
                 '\n'
                 '■ B. 수익성 관련 기초자료\n'
                 '\n'
                 '                              2023-12   2024-12   2025-12   동업계평균   추정1기\n'
                 '매출액증가율(%)                  15.97      15.25       8.27        3.06      13.16\n'
                 '매출원가율(%)                    90.57      87.93      84.53       84.45      84.23\n'
                 '영업이익률(%)                     2.77       5.43       8.69        6.71       9.03\n'
                 '금융비용부담률(%)                 2.29       2.48       1.67        2.09       1.48\n'
                 '이자보상배율(배)                  1.21       2.19       5.20        3.21       6.10\n'
                 '\n'
                 '[특이사항]\n'
                 '- 최근 3개년 연속 매출액 증가추세\n'
                 '- 수익성 개선 추세\n'
                 '\n'
                 '■ C. 재무안정성 및 자산의 질 관련 기초자료\n'
                 '\n'
                 '                              2023-12   2024-12   2025-12   동업계평균   추정1기\n'
                 '유동비율(%)                       82.89      85.41      94.53       93.63     105.93\n'
                 '부채비율(%)                      256.17     264.12     253.98      124.43     199.06\n'
                 '차입금의존도(%)                   29.59      33.45      27.93       24.75      26.35\n'
                 '비유동장기적합률(%)              131.47     127.69     110.26      104.30      90.84\n'
                 '평균매출채권/매출액(%)              0         16.10      14.54        0        16.85\n'
                 '평균재고자산/매출액(%)              0         33.46      39.62        0        35.95\n'
                 '유형자산/차입금(%)               105.87      87.57      87.93        0       104.24\n'
                 '\n'
                 '[특이사항]\n'
                 '- 재무안정성 지표 개선 추세\n'
                 '- 부채비율과 차입금의존도는 동업계 평균보다 높은 수준\n'
                 '\n'
                 '■ D. 현금흐름 및 채무상환능력 관련 기초자료\n'
                 '\n'
                 '[현금흐름표]\n'
                 '                                   2023년      2024년      2025년\n'
                 '영업활동으로 인한 현금흐름           -18,964      36,602      69,484\n'
                 '투자활동으로 인한 현금흐름             3,577     -14,621     -26,490\n'
                 '재무활동으로 인한 현금흐름             8,678      22,908     -12,688\n'
                 '기말 현금및현금성자산                  77,545     122,652     152,795\n'
                 '\n'
                 '[특이사항]\n'
                 '- 영업활동현금흐름 정(+)\n'
                 '- 2023년에는 매출채권, 재고자산 및 선급금 증가 등의 영향으로 영업활동현금흐름이 부(-)\n'
                 '- 2024년 영업활동현금흐름 정(+) 전환\n'
                 '- 2025년 영업활동현금흐름 69,484\n'
                 '\n'
                 '■ E. 주요 매출처 및 매출비중 관련 기초자료\n'
                 '\n'
                 '[2024년 주요 매출처]\n'
                 '한화에어로스페이스(주)      124,202 / 18.05%\n'
                 '방위사업청                   76,605 / 11.14%\n'
                 '한화오션(주)                 44,027 /  6.40%\n'
                 '에이치디현대중공업(주)       34,225 /  4.97%\n'
                 '현대로템(주)                 30,992 /  4.50%\n'
                 '상기 외                     377,908 / 54.93%\n'
                 '합계                        687,959 /100.00%\n'
                 '\n'
                 '[2025년 주요 매출처]\n'
                 '한화에어로스페이스(주)      152,364 / 20.46%\n'
                 '방위사업청                  109,015 / 14.64%\n'
                 '에스케이오션플랜트(주)       46,707 /  6.27%\n'
                 '엘아이지넥스원(주)           38,501 /  5.17%\n'
                 '(주)강남                    37,530 /  5.04%\n'
                 '상기 외                     360,702 / 48.43%\n'
                 '합계                        744,818 /100.00%\n'
                 '\n'
                 '[특이사항]\n'
                 '- 조선·방산 등 사업부문에서 고정매출처 외 다변화된 매출처를 대상으로 매출액 시현\n'
                 '- 한화에어로스페이스(주), 방위사업청, 엘아이지넥스원(주) 등 우량 매출처 확보\n'
                 '- 매출채권회전기간: 53.07일',
 'answers': {'A': '- 매출액 대비 매출채권 및 재고자산 과다한 점은 있으나, 2024년말 기준 재공품 121,995백만원, 미착원재료 58,030백만원으로 제작기간 장기간인 점 '
                  '등에 기인한 것으로, 당기중 감모손실 반영 금액은 없음.\n'
                  '\n'
                  '- 유산스 잔액에 따라 차입금 변동성 상존하는 가운데 2024년도 차입금은 유산스 증가 등으로 차입금 증가하였으며, 2025년도는 수주 증가에 따른 운전자금 '
                  '충당 목적 등으로 일반대 증가 등으로 증가하였으며, 총차입금은 증가 추세임.\n'
                  '\n'
                  '- 2025.03월 최대주주는 영구전환사채를 보통주로 전환하였고, 전환으로 인해 신종자본증권 94,351백만원 감소하고, 자본금이 31,230백만원, '
                  '주식발행초과금이 62,966백만원 증가함.\n'
                  '\n'
                  '- 2025.03월 자본잉여금으로 이익잉여금(결손금) 405,326백만원 보전하였고, 순이익 64,566백만원 자본전입 등으로 자본총계 증가함.',
             'B': '- 매출신장에 기반하여 매출원가율 및 영업이익률 등의 수익성 지표 개선 추세임.',
             'C': '- 동사 동업계평균 대비 부채비율 및 차입금의존도 다소 높은 편이나, 최근 3개년 지속 개선 추세에 있는 점 긍정적임.',
             'D': '- 2023년도 매출채권, 재고자산 및 선급금 증가 등으로 영업활동현금흐름 부(-) 시현하였으나, 2024년도 정(+) 전환하였으며, 2025년도 '
                  '영업활동현금흐름 69,484백만원으로 영업활동을 통한 현금창출능력 양호함.',
             'E': '- 한화에어로스페이스(주), 방위사업청, 엘아이지넥스원(주) 등 우량 매출처를 확보중인 가운데 매출채권회전기간(53.07일) 감안 매출채권 안정성 양호함.'}}


In [ ]:
# FEW SHOT 2 — 실제 우수 심사역 답안 기본값
# 모든 여신유형·업종에 일괄 적용하며, 사실이 아닌 문체·분석 구조로만 사용합니다.
FEW_SHOT_2 = {
    "case_summary": """
금액 단위는 백만원이며 2023~2025년 연결재무제표 기준임.

매출액은 2023년 420,000, 2024년 405,000, 2025년 360,000으로 감소함.
영업이익은 같은 기간 24,000, 8,000, -12,000이며 당기순이익은
15,000, 2,000, -18,000임.

매출채권은 75,000, 82,000, 90,000으로 증가했고 재고자산은
68,000, 85,000, 110,000으로 확대됨. 재고자산 중 제품과
재공품 비중이 상승했으나 장기체화 여부는 확인되지 않음.

영업활동현금흐름은 20,000, -5,000, -24,000임. 총차입금은
120,000, 155,000, 205,000으로 증가했으며 현금성자산은
35,000, 28,000, 20,000으로 감소함.

유동자산은 240,000, 유동부채는 270,000이며 2025년 말
1년 이내 만기도래 차입금은 95,000임. 자본총계는 2024년
180,000에서 2025년 160,000으로 감소함.

주요 거래처의 주문량 감소와 원재료 가격 상승이 실적 저하의
주요 원인으로 제공됨. 2026년 1분기 매출은 전년 동기 대비
4% 감소했으나 신규 거래처와의 공급계약 협의가 진행 중임.
계약 체결 여부와 예상 매출액은 현재 자료로 확정할 수 없음.
""".strip(),
    "answers": {
        "A": """
- 매출 감소에도 매출채권은 2023년 750억원에서 2025년 900억원으로 증가하여 외형 대비 채권성 자산의 자금 점유가 확대됨. 매출채권 증가가 실제 연체를 의미한다고 단정할 수는 없으나, 매출 감소와 반대 방향으로 움직인 점을 고려하면 회수조건과 채권 연령구조의 저하 가능성이 부담요인임. 채권 회수속도가 개선되지 않을 경우 운전자금 부담이 지속될 것으로 전망됨.

- 재고자산은 같은 기간 680억원에서 1,100억원으로 증가하여 매출 감소와 상반된 흐름을 보임. 제품과 재공품 비중이 상승한 가운데 영업활동현금흐름도 적자로 전환되어 재고 증가가 현금흐름을 제약한 것으로 판단됨. 장기체화 여부는 확정할 수 없으나 매출 회복이 지연될 경우 평가손실과 추가 운전자금 부담으로 이어질 가능성이 있음.

- 현금성자산이 감소하는 동안 총차입금은 1,200억원에서 2,050억원으로 증가하여 외부차입 의존도가 확대됨. 차입금 증가분이 영업손실과 운전자본 소요를 보전하는 데 사용된 것으로 추정되는 구조임. 영업현금흐름이 회복되지 않으면 차입금 증가세와 금융비용 부담이 지속될 것으로 판단됨.
""".strip(),  # 가. 재무제표 주요계정(현황 및 향후전망)
        "B": """
- 매출액은 2023년 4,200억원에서 2025년 3,600억원으로 감소했으며 영업이익은 240억원에서 120억원 적자로 전환됨. 주문량 감소와 원재료 가격 상승이 동시에 발생하여 외형 축소와 원가 부담 확대가 수익성을 함께 저하시킨 것으로 판단됨.

- 2025년 당기순손실은 180억원으로 영업적자와 금융부담이 최종 손익에 반영된 상태임. 신규 거래처 확보가 추진되고 있으나 계약 체결과 예상 매출이 확정되지 않아 단기적인 실적 회복의 근거로 반영하기는 어려움. 기존 거래처의 주문 회복 또는 원가 부담 완화가 확인되기 전까지 수익성 저하가 지속될 가능성이 높은 것으로 판단됨.
""".strip(),  # 나. 수익성(현황 및 향후전망)
        "C": """
- 영업손실과 당기순손실 발생으로 자본총계가 전년 1,800억원에서 1,600억원으로 감소한 반면 총차입금은 2,050억원으로 증가함. 손실 누적과 차입 확대가 동시에 나타나 재무완충력이 약화된 것으로 판단됨.

- 유동자산 2,400억원이 유동부채 2,700억원을 하회하여 유동비율은 100% 미만임. 현금성자산 감소와 단기 만기도래 차입금 950억원을 고려하면 단기 상환부담이 확대된 상태임. 매출채권과 재고자산의 회전이 개선되지 않을 경우 유동성 부담은 구조적으로 지속될 가능성이 있음.

- 재고자산과 매출채권이 확대된 가운데 손익과 현금흐름이 모두 저하되어 자산의 질에 대한 부담도 증가함. 다만 채권 연령과 재고 보유기간이 제공되지 않아 손상 또는 체화를 확정할 수는 없음. 현재 수치만으로도 영업자산의 현금전환 지연 가능성이 높아진 것으로 판단됨.
""".strip(),  # 다. 재무안정성 및 자산의 질(현황 및 향후전망)
        "D": """
- 영업활동현금흐름은 2023년 200억원 유입에서 2025년 240억원 유출로 전환됨. 영업적자와 매출채권·재고자산 증가가 함께 발생하여 손익 저하가 현금창출력 약화로 이어진 것으로 판단됨.

- 부족자금을 차입으로 보전하면서 총차입금은 3개년간 850억원 증가했으나 현금성자산은 150억원 감소함. 이에 따라 실질적인 순차입금 부담과 금융비용 부담이 동시에 확대된 상태임. 영업현금흐름이 흑자로 전환되지 않으면 자체 상환재원만으로 차입금을 축소하기 어려울 것으로 판단됨.

- 1년 이내 만기도래 차입금 950억원이 현금성자산 200억원을 크게 상회하여 차환 의존도가 높은 구조임. 구체적인 만기 일정과 금융기관 약정은 제공되지 않았으나, 현재의 현금창출력과 유동성 수준을 고려하면 단기 채무상환능력은 전년보다 저하된 것으로 판단됨.
""".strip(),  # 라. 현금흐름 및 채무상환능력(현황 및 향후전망)
        "E": """
- 주요 거래처의 주문량 감소가 전체 매출 하락으로 이어진 점을 고려하면 특정 거래처에 대한 매출 의존도가 높은 것으로 판단됨. 주요 거래처의 발주 변동이 외형과 생산설비 가동률에 직접적인 영향을 미치는 구조가 매출 안정성을 제약하고 있음.

- 신규 거래처와 공급계약을 협의하고 있으나 계약 체결과 예상 매출액이 확정되지 않아 현재 심사에서는 매출 회복요인으로 반영하기 어려움. 신규 거래가 실제 매출로 연결되기 전까지 기존 거래처의 주문 감소에 따른 매출 변동 위험이 지속될 것으로 전망됨.

- 향후 신규 거래처 매출이 발생하여 상위 거래처 비중이 낮아질 경우 매출처 편중 위험은 완화될 수 있음. 다만 현재 자료 기준으로는 거래처 다변화 효과보다 기존 거래처의 발주 감소에 따른 부정적 영향이 우세한 것으로 판단됨.
""".strip(),  # 마. 주요 매출처 및 매출비중 변동 추이
    },
}


In [ ]:
# FEW SHOT 3 — 실제 우수 심사역 답안 기본값
# 모든 여신유형·업종에 일괄 적용하며, 사실이 아닌 문체·분석 구조로만 사용합니다.
FEW_SHOT_3 = {
    "case_summary": """
금액 단위는 백만원이며 2023~2025년 연결재무제표 기준임.

매출액은 2023년 680,000, 2024년 650,000, 2025년 720,000임.
영업이익은 같은 기간 18,000, -6,000, 32,000이며 당기순이익은
8,000, -15,000, 20,000임.

매출채권은 110,000, 105,000, 112,000이며 재고자산은
95,000, 88,000, 92,000임. 계약자산은 25,000, 30,000,
45,000으로 증가함. 계약자산 증가의 구체적인 원인은
제공된 자료에서 확인되지 않음.

영업활동현금흐름은 12,000, -18,000, 48,000임. 유형·무형자산
취득액은 20,000, 35,000, 70,000으로 증가함.

총차입금은 210,000, 225,000, 250,000이며 현금성자산은
45,000, 30,000, 52,000임. 2025년 말 장기차입금 비중은
전년보다 상승함. 유동비율은 2024년 92%에서 2025년 108%로 개선됨.

자본총계는 2024년 190,000에서 2025년 235,000으로 증가함.
2025년 중 유상증자 20,000이 있었고 배당금 5,000이 지급됨.

2024년 주요 생산설비 정비로 가동률이 하락했으나 2025년 정상화됨.
2025년 설비투자 증가는 생산능력 확대와 노후설비 교체를 위한 것으로
제공됨. 2026년 1분기 매출은 전년 동기 대비 7% 증가함.

주요 매출처 상위 3개사의 매출 비중은 2023년 62%, 2024년 58%,
2025년 51%이며 신규 거래처 매출 비중은 2025년 12%임.
""".strip(),
    "answers": {
        "A": """
- 생산설비 정상화로 매출액은 2024년 6,500억원에서 2025년 7,200억원으로 회복되고 영업이익도 60억원 적자에서 320억원 흑자로 전환됨. 최근 분기 매출도 증가세를 유지하여 외형과 수익창출력의 회복 방향은 확인됨. 다만 회복기간이 아직 1개년에 불과하므로 현재 수준의 이익창출력이 지속되는지가 향후 판단의 주요 변수임.

- 매출채권과 재고자산은 매출 회복에도 전년 대비 제한적으로 증가하여 운전자본 부담이 외형보다 빠르게 확대되지는 않음. 반면 계약자산은 300억원에서 450억원으로 50% 증가하여 매출 증가율을 크게 상회함. 구체적인 발생 원인은 확인되지 않으나 청구 전 자산의 자금 점유가 확대된 점은 부담요인으로 판단됨.

- 생산능력 확대와 노후설비 교체를 위해 유형·무형자산 취득액이 350억원에서 700억원으로 증가함. 영업현금흐름 회복이 투자부담을 일부 흡수했으나 투자액이 영업현금 유입액을 상회하여 부족재원의 일부를 차입으로 조달한 구조임. 투자가 계획된 매출과 현금창출로 연결될 경우 중기적인 수익기반은 강화될 것으로 전망됨.
""".strip(),  # 가. 재무제표 주요계정(현황 및 향후전망)
        "B": """
- 2024년 생산설비 정비에 따른 가동률 저하로 영업적자가 발생했으나 2025년 정상화 이후 영업이익 320억원과 당기순이익 200억원을 기록함. 매출 회복과 생산 정상화가 수익성 개선으로 연결된 것으로 판단됨.

- 영업이익률은 2024년 적자에서 2025년 약 4.4%로 회복했으나 2023년과 비교한 개선폭은 아직 제한적임. 생산설비 정상화 효과가 온전히 반영되고 최근 분기 매출 증가세가 지속될 경우 추가적인 수익성 개선 가능성이 있음. 다만 신규 설비의 초기 고정비와 감가상각비 증가는 이익률 개선을 제약할 수 있음.

- 2025년 흑자 전환은 긍정적이나 단일 연도의 실적만으로 구조적인 수익성 회복을 확정하기는 어려움. 기존 생산설비의 안정적인 가동과 신규 투자설비의 매출 기여가 확인될 경우 회복의 지속 가능성이 높아질 것으로 판단됨.
""".strip(),  # 나. 수익성(현황 및 향후전망)
        "C": """
- 당기순이익 200억원과 유상증자 200억원이 배당금 50억원의 자본 유출을 상회하여 자본총계는 1,900억원에서 2,350억원으로 증가함. 이익 누적과 외부자본 확충이 동시에 이루어져 재무완충력은 전년보다 개선된 것으로 판단됨.

- 총차입금은 2,250억원에서 2,500억원으로 증가했으나 현금성자산도 300억원에서 520억원으로 확대됨. 이에 따라 차입금 증가액 대비 순차입금 증가폭은 제한적이며 장기차입금 비중 상승으로 만기구조도 일부 개선됨. 자본 확충을 고려하면 차입 증가에 따른 전반적인 재무구조 부담은 현 수준에서 감내 가능한 것으로 판단됨.

- 유동비율은 92%에서 108%로 개선되어 단기 지급능력은 전년보다 강화됨. 다만 계약자산과 설비투자에 대한 자금 점유가 증가한 점을 고려하면 영업현금흐름이 다시 약화될 경우 유동성 개선폭이 축소될 수 있음.
""".strip(),  # 다. 재무안정성 및 자산의 질(현황 및 향후전망)
        "D": """
- 영업활동현금흐름은 2024년 180억원 유출에서 2025년 480억원 유입으로 전환되어 손익 회복이 현금창출 개선으로 이어짐. 매출채권과 재고자산 증가가 제한적인 점도 현금전환력 회복에 기여한 것으로 판단됨.

- 설비 취득액 700억원이 영업현금흐름 480억원을 상회하여 투자 후 잉여현금흐름은 부족한 상태임. 이에 따라 총차입금이 250억원 증가했으나 현금성자산과 자본도 함께 확대되어 현재의 채무상환 부담은 급격히 악화되지 않은 것으로 판단됨.

- 장기차입금 비중 상승과 유동비율 개선은 단기 상환압력을 완화하는 요인임. 향후 신규 설비가 추가 매출과 영업현금흐름을 창출하면 차입금 상환능력도 개선될 수 있음. 반대로 투자성과가 지연될 경우 감가상각비와 금융비용이 현금흐름을 제약할 가능성이 있음.
""".strip(),  # 라. 현금흐름 및 채무상환능력(현황 및 향후전망)
        "E": """
- 상위 3개 매출처의 비중은 2023년 62%에서 2025년 51%로 하락하여 주요 거래처에 대한 매출 편중이 점진적으로 완화됨. 신규 거래처 매출이 2025년 전체 매출의 12%를 차지한 점을 고려하면 거래처 다변화가 실제 매출구조 변화로 이어진 것으로 판단됨.

- 상위 매출처 비중은 낮아졌으나 여전히 전체 매출의 절반을 상회하여 주요 거래처의 발주 변동이 실적에 미치는 영향은 상당한 수준임. 다만 신규 거래처 매출이 안정적으로 유지될 경우 기존 거래처 의존에 따른 변동성은 추가로 완화될 것으로 전망됨.

- 거래처 다변화와 전체 매출 회복이 동시에 나타나 매출기반의 안정성은 전년보다 개선된 것으로 판단됨. 향후 신규 거래처의 반복매출 여부와 상위 거래처 비중의 추가 하락이 매출 안정성 개선을 판단하는 핵심 요소임.
""".strip(),  # 마. 주요 매출처 및 매출비중 변동 추이
    },
}


In [ ]:
ngrok_token = secret("NGROK_AUTHTOKEN")
if not ngrok_token:
    raise RuntimeError("Colab Secrets의 NGROK_AUTHTOKEN이 필요합니다")
hf_token = secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Colab Secrets의 HF_TOKEN이 필요합니다")

import os
import secrets
from fastapi.responses import FileResponse
from importlib.resources import files
from IPython.display import HTML, display
from pyngrok import ngrok
from urllib.request import Request
import uvicorn

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Gemma 4 MoE 운영에는 Colab GPU 런타임이 필요합니다")

# 모델 다운로드 전에 ngrok 엔드포인트 중복 여부를 확인합니다. Basic Auth는 사용하지 않습니다.
public_url = None
ngrok.set_auth_token(ngrok_token)
try:
    ngrok.kill()
except Exception:
    pass
try:
    tunnel = ngrok.connect(addr=PORT, proto="http", bind_tls=True)
    public_url = tunnel.public_url
    log(f"ngrok endpoint reserved: {public_url}")
except Exception as exc:
    raise RuntimeError(
        "ngrok 엔드포인트가 이미 사용 중입니다. 기존 Colab 런타임을 종료한 뒤 다시 실행하세요."
    ) from exc

# 세 입력 셀을 심사항목별 3-shot(JSON 15개 레코드)으로 변환합니다.
few_shot_rows = []
item_titles = {
    "A": "재무제표 주요계정(현황 및 향후전망)",
    "B": "수익성(현황 및 향후전망)",
    "C": "재무안정성 및 자산의 질(현황 및 향후전망)",
    "D": "현금흐름 및 채무상환능력(현황 및 향후전망)",
    "E": "주요 매출처 및 매출비중 변동 추이",
}
for shot_number, shot in enumerate((FEW_SHOT_1, FEW_SHOT_2, FEW_SHOT_3), start=1):
    summary = str(shot.get("case_summary") or "").strip()
    for item_code, title in item_titles.items():
        answer = str((shot.get("answers") or {}).get(item_code) or "").strip()
        if answer:
            few_shot_rows.append({
                "example_id": f"COLAB-FS{shot_number}-{item_code}",
                "review_item_code": item_code,
                "input_summary": summary,
                "output_example": answer,
                "example_version": "1",
                "approval_status": "APPROVED",
                "loan_types": [],
                "industry_codes": [],
                "situation_tags": [],
                "style_tags": ["expert-reviewer", "global-application", title],
            })
few_shot_path = stage_root / "few_shots_runtime.json"
few_shot_path.write_text(
    json.dumps({"examples": few_shot_rows}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
log(f"few-shot loaded: {len(few_shot_rows)} item examples (3 cases × A-E)")

log("installing isolated vLLM GPU runtime")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    "uv",
], check=True)
uv_command = shutil.which("uv") or "uv"
vllm_env = stage_root / "vllm-env"
subprocess.run([
    uv_command, "venv", "--python", sys.executable, "--seed", str(vllm_env),
], check=True)
vllm_python = vllm_env / "bin" / "python"
vllm_executable = vllm_env / "bin" / "vllm"
ninja_executable = vllm_env / "bin" / "ninja"
subprocess.run([
    uv_command, "pip", "install", "--python", str(vllm_python),
    "-U", "vllm", "sentencepiece", "protobuf", "ninja",
    "--pre",
    "--extra-index-url", "https://wheels.vllm.ai/nightly/cu129",
    "--extra-index-url", "https://download.pytorch.org/whl/cu129",
    "--index-strategy", "unsafe-best-match",
], check=True)
if not vllm_executable.is_file():
    raise RuntimeError(f"isolated vLLM executable is missing: {vllm_executable}")
if not ninja_executable.is_file():
    raise RuntimeError(f"isolated Ninja executable is missing: {ninja_executable}")
ninja_probe = subprocess.run(
    [str(ninja_executable), "--version"],
    capture_output=True,
    text=True,
)
if ninja_probe.returncode != 0:
    raise RuntimeError(
        "isolated Ninja validation failed before model startup:\n"
        + (ninja_probe.stderr or ninja_probe.stdout)[-4000:]
    )
log(f"vLLM build tool ready · ninja {ninja_probe.stdout.strip()}")

# vLLM 환경은 별도 프로세스에 격리한다. E5가 사용할 현재 커널의 NumPy,
# SciPy, torch, transformers는 변경하지 않으며 모델 다운로드 전에 검사한다.
dependency_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import numpy, torch; "
            "from transformers import AutoModel, AutoTokenizer; "
            "print(numpy.__version__, torch.__version__)"
        ),
    ],
    capture_output=True,
    text=True,
)
if dependency_probe.returncode != 0:
    raise RuntimeError(
        "E5 dependency validation failed before model download:\n"
        + (dependency_probe.stderr or dependency_probe.stdout)[-4000:]
    )
log("E5 dependency stack ready · vLLM environment isolated")

MODEL_ID = "google/gemma-4-26B-A4B-it"
VLLM_PORT = 8001
vllm_api_key = secrets.token_urlsafe(24)
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
vllm_log_path = stage_root / "vllm.log"
vllm_log_handle = vllm_log_path.open("w", encoding="utf-8")
vllm_process_env = os.environ.copy()
vllm_process_env["PATH"] = (
    str(vllm_env / "bin")
    + os.pathsep
    + vllm_process_env.get("PATH", "")
)
vllm_process = subprocess.Popen(
    [
        str(vllm_executable),
        "serve", MODEL_ID,
        "--host", "127.0.0.1",
        "--port", str(VLLM_PORT),
        "--served-model-name", MODEL_ID,
        "--api-key", vllm_api_key,
        "--dtype", "bfloat16",
        "--model-impl", "vllm",
        "--gpu-memory-utilization", "0.90",
        "--max-model-len", "16384",
        "--max-num-seqs", "4",
        "--max-num-batched-tokens", "8192",
        "--enable-prefix-caching",
        "--async-scheduling",
        "--limit-mm-per-prompt", json.dumps({"image": 0, "audio": 0}, separators=(",", ":")),
    ],
    stdout=vllm_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    env=vllm_process_env,
)
log(f"loading native vLLM: {MODEL_ID} · A100 BF16 · concurrent sequences=4")
vllm_ready = False
for _ in range(1800):
    if vllm_process.poll() is not None:
        vllm_log_handle.flush()
        full_log = vllm_log_path.read_text(encoding="utf-8", errors="replace")
        marker = full_log.find("EngineCore failed to start")
        if marker >= 0:
            start = max(0, marker - 2000)
            excerpt = full_log[start : start + 30000]
            if start + 30000 < len(full_log):
                excerpt += "\n... [middle omitted] ...\n" + full_log[-4000:]
        else:
            excerpt = full_log[-30000:]
        raise RuntimeError(
            f"vLLM startup failed. Full log: {vllm_log_path}\n" + excerpt
        )
    try:
        call = Request(
            f"http://127.0.0.1:{VLLM_PORT}/v1/models",
            headers={"Authorization": f"Bearer {vllm_api_key}"},
        )
        with urlopen(call, timeout=2) as response:
            if response.status == 200:
                vllm_ready = True
                break
    except (URLError, TimeoutError, OSError):
        pass
    time.sleep(1)
if not vllm_ready:
    raise RuntimeError("vLLM did not become healthy within 30 minutes")
log("vLLM ready · streaming generation · 1400 tokens + automatic continuation")

# vLLM 설치가 끝난 뒤 현재 커널에서 애플리케이션을 처음 import합니다.
from semantic_prompt_transfer import (
    E5GpuEncoder,
    OpenAICompatibleHttpGenerator,
    RemoteGenerationConfig,
    __version__,
    build_colab_poc,
)

if __version__ != PACKAGE_VERSION:
    raise RuntimeError(f"installed package version mismatch: {__version__}")

# A100의 남은 메모리에서 작은 E5를 상주시켜 임베딩을 GPU 배치 처리합니다.
embedding_encoder = E5GpuEncoder(
    token=hf_token,
    batch_size=128,
    max_length=384,
    stride=32,
)
local_generator = OpenAICompatibleHttpGenerator(
    RemoteGenerationConfig(
        base_url=f"http://127.0.0.1:{VLLM_PORT}/v1",
        model=MODEL_ID,
        api_key=vllm_api_key,
        timeout_seconds=300,
        max_new_tokens=1400,
        max_continuations=2,
        temperature=0.0,
        allow_insecure_http=True,
    )
)

bundle = None
server = None
server_thread = None
try:
    bundle = build_colab_poc(
        model_dir=stage_root,
        root=runtime_root,
        few_shot_path=few_shot_path,
        encoder=embedding_encoder,
        generator=local_generator,
        demo_credit_report_path=DEMO_CREDIT_REPORT,
        demo_attachment_paths=DEMO_ATTACHMENTS,
        anonymous_access=True,
    )
    app = bundle.app
    html_path = files("semantic_prompt_transfer.examples.operational").joinpath(
        "credit_review_upload_demo.html"
    )

    @app.get("/", include_in_schema=False)
    def poc_screen():
        return FileResponse(str(html_path), media_type="text/html")

    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
    server = uvicorn.Server(config)
    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()

    health = None
    for _ in range(60):
        try:
            with urlopen(f"http://127.0.0.1:{PORT}/api/v1/runtime/health", timeout=2) as response:
                if response.status == 200:
                    health = json.loads(response.read().decode("utf-8"))
                    break
        except (URLError, TimeoutError, OSError):
            pass
        time.sleep(1)
    if health is None:
        raise RuntimeError("POC API did not become healthy within 60 seconds")

    query = urlencode({"mode": "api", "api_base": public_url})
    launch_url = f"{public_url}/?{query}"

    def cleanup() -> None:
        global bundle, server, server_thread, public_url, vllm_process, vllm_log_handle
        if public_url:
            try:
                ngrok.disconnect(public_url)
            except Exception:
                pass
            public_url = None
        if server is not None:
            server.should_exit = True
        if server_thread is not None and server_thread.is_alive():
            server_thread.join(timeout=10)
        if bundle is not None:
            try:
                bundle.close()
            finally:
                bundle = None
        if vllm_process is not None and vllm_process.poll() is None:
            vllm_process.terminate()
            try:
                vllm_process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                vllm_process.kill()
        if not vllm_log_handle.closed:
            vllm_log_handle.close()

    globals()["_SPT_COLAB_CLEANUP"] = cleanup
    atexit.register(cleanup)
    log(f"ready: package={__version__}, llm=vLLM/{MODEL_ID}, embedding=GPU E5, anonymous POC")
    log("multi-user routing: random browser scope + vLLM continuous batching (max 4)")
    display(
        HTML(
            "<h3>SemanticPromptTransfer POC가 준비되었습니다.</h3>"
            f"<p><a href='{launch_url}' target='_blank' rel='noopener'>심사 화면 바로 열기</a></p>"
            "<p>별도 로그인·공통 비밀번호가 없습니다. URL을 아는 사람은 접속할 수 있으며, "
            "Colab 종료 시 업로드·벡터·임시 ID가 삭제됩니다.</p>"
        )
    )
except Exception:
    if public_url:
        try:
            ngrok.disconnect(public_url)
        except Exception:
            pass
    if server is not None:
        server.should_exit = True
    if server_thread is not None and server_thread.is_alive():
        server_thread.join(timeout=10)
    if bundle is not None:
        bundle.close()
    if vllm_process is not None and vllm_process.poll() is None:
        vllm_process.terminate()
    if not vllm_log_handle.closed:
        vllm_log_handle.close()
    raise
